In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

In [2]:
class Generator(nn.Module):
    def __init__(self, z_dim=100, num_classes=3, img_channels=1):
        super().__init__()
        self.label_embed = nn.Embedding(num_classes, z_dim)
        self.gen = nn.Sequential(
            nn.ConvTranspose2d(z_dim * 2, 1024, 4, 1, 0), 
            nn.BatchNorm2d(1024),
            nn.ReLU(True),
            nn.ConvTranspose2d(1024, 512, 4, 2, 1), 
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, img_channels, 4, 2, 1),
            nn.Tanh() 
        )

    def forward(self, noise, labels):
        label_vec = self.label_embed(labels).unsqueeze(2).unsqueeze(3)
        x = torch.cat([noise, label_vec], dim=1)
        return self.gen(x)

In [3]:
class Discriminator(nn.Module):
    def __init__(self, num_classes=3, img_channels=1, img_size=128):
        super().__init__()
        self.img_size = img_size
        self.label_embed = nn.Embedding(num_classes, img_size * img_size)
        self.disc = nn.Sequential(
            nn.Conv2d(img_channels + 1, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.InstanceNorm2d(128, affine=True), 
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.InstanceNorm2d(256, affine=True),
            nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 4, 2, 1),
            nn.InstanceNorm2d(512, affine=True),
            nn.LeakyReLU(0.2),
            nn.Conv2d(512, 1024, 4, 2, 1),
            nn.InstanceNorm2d(1024, affine=True),
            nn.LeakyReLU(0.2),
            nn.Conv2d(1024, 1, 4, 1, 0), 
        )

    def forward(self, x, labels):
        label_mask = self.label_embed(labels).view(-1, 1, self.img_size, self.img_size)
        x = torch.cat([x, label_mask], dim=1)
        return self.disc(x)

In [4]:
def gradient_penalty(critic, labels, real, fake, device):
    BATCH_SIZE, C, H, W = real.shape
    alpha = torch.rand((BATCH_SIZE, 1, 1, 1)).repeat(1, C, H, W).to(device)
    interpolated_images = (real * alpha + fake * (1 - alpha)).requires_grad_(True)
    mixed_scores = critic(interpolated_images, labels)
    gradient = torch.autograd.grad(
        inputs=interpolated_images,
        outputs=mixed_scores,
        grad_outputs=torch.ones_like(mixed_scores),
        create_graph=True,
        retain_graph=True,
    )[0]
    gradient = gradient.view(gradient.shape[0], -1)
    gradient_norm = gradient.norm(2, dim=1)
    return torch.mean((gradient_norm - 1) ** 2)

In [ ]:
def save_labeled_samples(gen, noise, labels, epoch, device):
    gen.eval()
    class_names = [" COVID19 ", " NORMAL ", " PNEUMONIA "]
    with torch.no_grad():
        fake = gen(noise, labels)
        img_grid = vutils.make_grid(fake, nrow=8, normalize=True).cpu()
        
        plt.figure(figsize=(10, 5))
        plt.imshow(img_grid.permute(1, 2, 0), cmap='gray')
        plt.axis('off')
        
        # Add text labels on the left side for each row
        for i, name in enumerate(class_names):
            plt.text(-50, 64 + (i * 130), name, fontsize=8, fontweight='bold', rotation=90, va='center')
            
        plt.title(f"Generated X-Rays - Epoch {epoch}")
        plt.savefig(f"output_images/epoch_{epoch}.png", bbox_inches='tight')
        plt.close()
    gen.train()

In [ ]:
def train():
    from utils.dataset import get_dataloader 
    device = "cuda" if torch.cuda.is_available() else "cpu"
    z_dim, lr, batch_size, lambda_gp, num_epochs = 100, 1e-4, 64, 10, 51

    gen = Generator(z_dim).to(device)
    critic = Discriminator().to(device)
    opt_gen = optim.Adam(gen.parameters(), lr=lr, betas=(0.0, 0.9))
    opt_critic = optim.Adam(critic.parameters(), lr=lr, betas=(0.0, 0.9))

    loader, _ = get_dataloader("C:/Users/deep/Documents/DataSet/IAI/data/train", batch_size=batch_size)

    os.makedirs("saved_models", exist_ok=True)
    os.makedirs("output_images", exist_ok=True)
    
    # 8 samples per class for visualization
    fixed_noise = torch.randn(24, z_dim, 1, 1).to(device)
    fixed_labels = torch.tensor([0]*8 + [1]*8 + [2]*8).to(device)

    for epoch in range(num_epochs):
        loop = tqdm(loader, leave=True)
        for batch_idx, (real, labels) in enumerate(loop):
            real, labels = real.to(device), labels.to(device)
            cur_batch_size = real.shape[0]

            for _ in range(5):
                noise = torch.randn(cur_batch_size, z_dim, 1, 1).to(device)
                fake = gen(noise, labels)
                critic_real = critic(real, labels).reshape(-1)
                critic_fake = critic(fake.detach(), labels).reshape(-1)
                gp = gradient_penalty(critic, labels, real, fake.detach(), device)
                loss_critic = (torch.mean(critic_fake) - torch.mean(critic_real)) + (lambda_gp * gp)
                critic.zero_grad()
                loss_critic.backward()
                opt_critic.step()

            gen_fake = critic(fake, labels).reshape(-1)
            loss_gen = -torch.mean(gen_fake)
            gen.zero_grad()
            loss_gen.backward()
            opt_gen.step()

            loop.set_description(f"Epoch [{epoch}/{num_epochs}]")
            loop.set_postfix(loss_c=loss_critic.item(), loss_g=loss_gen.item())

        if epoch % 5 == 0:
            save_labeled_samples(gen, fixed_noise, fixed_labels, epoch, device)
            torch.save(gen.state_dict(), f"saved_models/gen_e{epoch}.pth")

if __name__ == "__main__":
    train()

Epoch [6/51]:  30%|██▉       | 24/81 [01:21<01:14,  1.31s/it, loss_c=-17.4, loss_g=20.6] 